In [ ]:
!git clone https://github.com/YPolina/Medicine.git

In [ ]:
%cd ./Medicine/BELKA/training

In [ ]:
!pip install -r ../requirements.txt

In [ ]:
import sys
import h5py
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd

sys.path.append(os.path.abspath(os.path.join('..')))

sys.modules.pop("functionality.models", None)
sys.modules.pop("functionality.data_preparation", None)
from functionality.models import CNNBinaryClassifierLightning
from functionality.data_preparation import IterfeaturesDataset

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from transformers import AutoTokenizer, AutoModel

import torch
import pickle
from tqdm import tqdm
import numpy as np
import gc
from torch.cuda.amp import autocast
from fastparquet import write

/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def compute_and_save_encoded_smiles(smiles, labels, save_path, batch_size=1000, max_length=142):
    """
    Compute encoded SMILES representations in batches and save them dynamically using Parquet

    Args:
        smiles (pd.Series): Data containing SMILES strings
        labels (pd.Series): Corresponding labels
        save_path (str): Path to save computed encoded SMILES and labels in Parquet format
        batch_size (int): Number of SMILES strings to process per batch
        max_length (int): Maximum length of encoded SMILES strings
    """
    # Encoding dictionary
    enc_dict = {
        'l': 1, 'y': 2, '@': 3, '3': 4, 'H': 5, 'S': 6, 'F': 7, 'C': 8, 'r': 9, 's': 10, '/': 11, 'c': 12, 'o': 13,
        '+': 14, 'I': 15, '5': 16, '(': 17, '2': 18, ')': 19, '9': 20, 'i': 21, '#': 22, '6': 23, '8': 24, '4': 25, '=': 26,
        '1': 27, 'O': 28, '[': 29, 'D': 30, 'B': 31, ']': 32, 'N': 33, '7': 34, 'n': 35, '-': 36
    }

    def encode_smile(smile):
        """Encodes a SMILES string to numerical values and pads it"""
        encoded = [enc_dict.get(char, 0) for char in smile]
        padded = encoded + [0] * (max_length - len(encoded))
        return padded

    for i in tqdm(range(0, len(smiles), batch_size), desc="Encoding SMILES"):
        batch_smiles = smiles.iloc[i : i + batch_size].tolist()
        batch_labels = labels.iloc[i : i + batch_size].values.astype(np.int16)

        batch_encoded_smiles = np.array([encode_smile(smile) for smile in batch_smiles], dtype=np.int64)

        df_batch = pd.DataFrame(batch_encoded_smiles, columns=[f"char_{j}" for j in range(max_length)])
        df_batch["label"] = batch_labels

        try:
            write(save_path, df_batch, append=True)
        except FileNotFoundError:
            write(save_path, df_batch)

        del batch_smiles, batch_labels, batch_encoded_smiles, df_batch
        gc.collect()

    print(f"Encoded SMILES and labels dynamically saved to {save_path}")
    return save_path

In [10]:
protein_names = ['sEH', 'BRD4', 'HSA']
save_dir = '../intermediates/embeddings/'
model_name = 'CNN'
batch_size=1000


for protein_name in protein_names:
    print(f"Smiles encoding for protein: {protein_name}")

    train_data = pd.read_parquet(f'../intermediates/train_data/{protein_name}/{protein_name}_train.parquet')
    val_data = pd.read_parquet(f'../intermediates/train_data/{protein_name}/{protein_name}_val.parquet')

    train_embeddings_path = os.path.join(save_dir, f"{protein_name}_{model_name}_train.parquet")
    val_embeddings_path = os.path.join(save_dir, f"{protein_name}_{model_name}_val.parquet")

    compute_and_save_encoded_smiles(train_data["molecule_smiles"], train_data['binds'], save_path=train_embeddings_path)
    compute_and_save_encoded_smiles(val_data["molecule_smiles"], val_data['binds'], save_path=val_embeddings_path)


Smiles encoding for protein: sEH


Encoding SMILES: 100%|██████████| 3648/3648 [09:19<00:00,  6.53it/s]


Encoded SMILES and labels saved to ../intermediates/embeddings/sEH_CNN_train.h5


Encoding SMILES: 100%|██████████| 406/406 [00:57<00:00,  7.02it/s]


Encoded SMILES and labels saved to ../intermediates/embeddings/sEH_CNN_val.h5
Smiles encoding for protein: BRD4


...

Encoded SMILES and labels saved to ../intermediates/embeddings/BRD4_CNN_val.h5
Smiles encoding for protein: HSA


Encoding SMILES: 100%|██████████| 3370/3370 [08:09<00:00,  6.89it/s]


Encoded SMILES and labels saved to ../intermediates/embeddings/HSA_CNN_train.h5


Encoding SMILES: 100%|██████████| 375/375 [00:54<00:00,  6.88it/s]

Encoded SMILES and labels saved to ../intermediates/embeddings/HSA_CNN_val.h5


In [ ]:
#aws
def train_model(model_name, protein_names, emb_path='s3://smiles-processing-py/embeddings/'):
    for protein_name in protein_names:

        print(f"Training model for protein: {protein_name}")

        train_embeddings_path = f'{emb_path}{protein_name}_{model_name}_train.parquet'
        val_embeddings_path = f'{emb_path}{protein_name}_{model_name}_val.parquet'
        
        train_dataset = IterfeaturesDataset(train_embeddings_path)
        train_loader = DataLoader(train_dataset, batch_size=1000, num_workers=4)

        val_dataset = IterfeaturesDataset(val_embeddings_path)
        val_loader = DataLoader(val_dataset, batch_size=1000, num_workers=4)

        logger = CSVLogger("logs", name=model_name)
        early_stopping = EarlyStopping(monitor="val_loss", patience=2, mode="min")
        checkpoint_callback = ModelCheckpoint(
            dirpath='s3://smiles-processing-py/checkpoints/',
            filename=f"{model_name}_{protein_name}-{{epoch}}-{{val_loss:.4f}}",
            monitor="val_loss",
            save_top_k=1,
            mode="min",
            save_last=True,
            verbose=True,
        )

        trainer = pl.Trainer(
            max_epochs=10,
            accelerator="auto",
            devices=1,
            log_every_n_steps=2,
            callbacks=[early_stopping, checkpoint_callback],
            logger=logger,
        )


        chemberta_model = ChemBertaBinaryClassifierLightning()
        trainer.fit(chemberta_model, train_loader, val_loader)

        os.makedirs("s3://smiles-processing-py/models", exist_ok=True)
        trainer.save_checkpoint(f"s3://smiles-processing-py/models/{model_name}_{protein_name}.ckpt")

        print(f"Completed training for protein: {protein_name}")

        del train_data, val_data, train_dataset, val_dataset, train_loader, val_loader, chemberta_model
        gc.collect()

In [2]:
def train_model(model_name, protein_names, emb_path="../intermediates/embeddings"):
    for protein_name in protein_names:

        print(f"Training model for protein: {protein_name}")

        train_embeddings_path = os.path.join(emb_path, f"{protein_name}_{model_name}_train.parquet")
        val_embeddings_path = os.path.join(emb_path, f"{protein_name}_{model_name}_val.parquet")
        
        train_dataset = IterfeaturesDataset(train_embeddings_path)
        train_loader = DataLoader(train_dataset, batch_size=1000, num_workers=4)

        val_dataset = IterfeaturesDataset(val_embeddings_path)
        val_loader = DataLoader(val_dataset, batch_size=1000, num_workers=4)

        logger = CSVLogger("logs", name=model_name)
        early_stopping = EarlyStopping(monitor="val_loss", patience=3, mode="min")
        checkpoint_callback = ModelCheckpoint(
            dirpath="../checkpoints",
            filename=f"{model_name}_{protein_name}-{{epoch}}-{{val_loss:.4f}}",
            monitor="val_loss",
            save_top_k=1,
            mode="min",
            save_last=True,
            verbose=True,
        )

        trainer = pl.Trainer(
            max_epochs=10,
            accelerator="auto",
            devices=1,
            log_every_n_steps=2,
            callbacks=[early_stopping, checkpoint_callback],
            logger=logger,
        )


        chemberta_model = CNNBinaryClassifierLightning()
        trainer.fit(chemberta_model, train_loader, val_loader)

        os.makedirs("../intermediates/models", exist_ok=True)
        trainer.save_checkpoint(f"../intermediates/models/{model_name}_{protein_name}.ckpt")

        print(f"Completed training for protein: {protein_name}")

        del train_data, val_data, train_dataset, val_dataset, train_loader, val_loader, chemberta_model
        gc.collect()

In [3]:
train_model('CNN', ['HSA'])

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Training model for protein: HSA


/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/user/Desktop/Pharm/Medicine/BELKA/checkpoints exists and is not empty.

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | loss_fn   | CrossEntropyLoss | 0      | train
1 | auroc     | BinaryAUROC      | 0      | train
2 | conv1     | Conv1d           | 128    | train
3 | conv2     | Conv1d           | 6.2 K  | train
4 | dropout   | Dropout          | 0      | train
5 | leakyrelu | LeakyReLU        | 0      | train
6 | sigmoid   | Sigmoid          | 0      | train
7 | pool      | MaxPool1d        | 0      | train
8 | fc1       | Linear           | 286 K  | train
9 | fc2       | Linear           | 258    | train
-------------------------------------------------------
293 K     Trainable params
0         Non-trainable params
293 K     Total params
1.174     Total estimated model

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/pytorch_lightning/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.


...

Epoch 3: 100%|██████████| 3370/3370 [18:29<00:00,  3.04it/s, v_num=6, val_loss=0.253, val_auc=0.844, train_loss=0.251, train_auc=0.833]

Epoch 3, global step 13480: 'val_loss' reached 0.25300 (best 0.25300), saving model to '/home/user/Desktop/Pharm/Medicine/BELKA/checkpoints/CNN_HSA-epoch=3-val_loss=0.2530.ckpt' as top 1


Epoch 4:  64%|██████▍   | 2169/3370 [12:11<06:44,  2.97it/s, v_num=6, val_loss=0.253, val_auc=0.844, train_loss=0.251, train_auc=0.833]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined